In [1]:
# Library
#!pip install timm
import os
import json
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

In [2]:
# Device agnostic code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Path
data_path = "/content/drive/MyDrive/Class/ESTSOFT/project/unstructured_data_analysis/data/downloaded"

In [4]:
# Hyperparameters
label_map = {"anger": 0, "happy": 1, "panic": 2, "sadness": 3}
num_labels = len(label_map)
image_size = 224
batch_size = 32
epochs = 5

In [5]:
# Load json
def load_json(file):
	with open(file=file, mode='r', encoding='euc-kr') as f:
		return json.load(f)

In [27]:
# Dataset
class FaceDataset(Dataset):
	def __init__(self, dir_root, set_name, transform=None):
		self.samples = []
		self.dir_root = dir_root
		self.set_name = set_name  # train, val, test
		self.transform = transform

		# Set paths
		if set_name == "test":
			label_dir = os.path.join(dir_root, "test", "label")
			img_dir_root = os.path.join(dir_root, "test", "image")
		else:
			label_dir = os.path.join(dir_root, "train", "label", set_name)
			img_dir_root = os.path.join(dir_root, "train", "image", set_name)

		# Iterate through the directory to grab information about images
		label_paths = [os.path.join(label_dir, f) for f in os.listdir(label_dir) if f.endswith(".json")]
		for label_path in label_paths:
			label = os.path.basename(label_path).split("_")[-1].split(".")[0]  # "train_anger.json" -> "anger"
			img_dir = os.path.join(img_dir_root, label)

			data = load_json(label_path)
			for item in data:
				image_path = os.path.join(img_dir, item["filename"])

				# Check if the file actually exists
				if not os.path.exists(image_path):
					continue

				self.samples.append({"image_path": os.path.join(img_dir, item["filename"]),
						 			 "label": label,
									 "bbox": item["annot_A"]["boxes"]})

	def __len__(self):
		return len(self.samples)

	def __getitem__(self, idx):
		item = self.samples[idx]

		image = Image.open(item["image_path"]).convert("RGB")
		if self.transform:
			image = self.transform(image) # C, W, H
		width, height = image.shape[2], image.shape[1] # Required for normalisation

		bbox = item["bbox"]
		X_min = bbox["minX"] / width
		Y_min = bbox["minY"] / height
		X_max = bbox["maxX"] / width
		Y_max = bbox["maxY"] / height
		bbox = torch.tensor([X_min, Y_min, X_max, Y_max], dtype=torch.float32)

		label = torch.zeros(num_labels, dtype=torch.float32)
		label[label_map[item["label"]]] = 1.0 # One-hot encoding

		targets = {"bbox": bbox, "label": label}

		return image, targets

In [28]:
# Transform
transform = transforms.Compose([transforms.Resize((image_size, image_size)),
                                transforms.ToTensor()])

In [29]:
# Dataset
ds_train = FaceDataset(dir_root=data_path, set_name="train", transform=transform)
ds_val   = FaceDataset(dir_root=data_path, set_name="val", transform=transform)
ds_test  = FaceDataset(dir_root=data_path, set_name="test", transform=transform)


In [30]:
# Dataloader
dl_train = DataLoader(dataset=ds_train, batch_size=batch_size, shuffle=True)
dl_val = DataLoader(dataset=ds_val, batch_size=batch_size, shuffle=False)
dl_test = DataLoader(dataset=ds_test, batch_size=batch_size, shuffle=False)

In [31]:
# Visual transformer model
class ViTModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model("vit_base_patch16_224", pretrained=True)
        self.num_features = self.backbone.head.in_features
        self.backbone.head = nn.Identity()
        self.fc_bbox = nn.Linear(self.num_features, 4)
        self.fc_label = nn.Linear(self.num_features, num_classes)

    def forward(self, x):
        z = self.backbone(x)
        bbox = self.fc_bbox(z)
        label = self.fc_label(z)
        return bbox, label


In [11]:
# Instantiation
model = ViTModel(num_classes=num_labels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_bbox = nn.SmoothL1Loss()
loss_label = nn.BCEWithLogitsLoss()

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

In [ ]:
for epoch in range(epochs):
    # Training
    model.train()
    loss_train_total = 0
    for X_train, y_train in dl_train:
        X_train = X_train.to(device)
        y_train_bbox = y_train["bbox"].to(device)
        y_train_label = y_train["label"].to(device)

        y_train_pred_bbox, y_train_pred_label = model(X_train)
        loss_train_bbox = loss_bbox(y_train_pred_bbox, y_train_bbox)
        loss_train_label = loss_label(y_train_pred_label, y_train_label)
        loss_train = loss_train_bbox + loss_train_label

        optimizer.zero_grad()
        loss_train.backward()
        optimizer.step()

        loss_train_total += loss_train.item()

    # Validation
    model.eval()
    loss_val_total = 0
    with torch.inference_mode():
        for X_val, y_val in dl_val:
            X_val = X_val.to(device)
            y_val_bbox = y_val["bbox"].to(device)
            y_val_label = y_val["label"].to(device)

            y_val_pred_bbox, y_val_pred_label = model(X_val)
            loss_val_bbox = loss_bbox(y_val_pred_bbox, y_val_bbox)
            loss_val_label = loss_label(y_val_pred_label, y_val_label)
            loss_val = loss_val_bbox + loss_val_label

            loss_val_total += loss_val.item()

    print(f"Epoch {epoch+1} | Train Loss: {loss_train_total / len(dl_train):.4f} | Val Loss: {loss_val_total / len(dl_val):.4f}")

    torch.save(model.state_dict(), os.path.join(data_path, "vit.pth"))
    torch.save(model, os.path.join(data_path, "vit.pt"))

In [24]:
# Testing
model.eval()
loss_test_total = 0
with torch.inference_mode():
    for X_test, y_test in dl_test:
        X_test = X_test.to(device)
        y_test_bbox = y_test["bbox"].to(device)
        y_test_label = y_test["label"].to(device)

        y_test_pred_bbox, y_test_pred_label = model(X_test)
        loss_test_bbox = loss_bbox(y_test_pred_bbox, y_test_bbox)
        loss_test_label = loss_label(y_test_pred_label, y_test_label)
        loss_test = loss_test_bbox + loss_test_label

        loss_test_total += loss_test.item()

print(f"\nTest Loss: {loss_test_total / len(dl_test):.4f}")

KeyboardInterrupt: 